In [ ]:
demo_notebook.py
# ================================================================
# SmartWardrobe — Demo Setup & Run
# Copy each cell into your Colab notebook
# ================================================================
 
 


In [ ]:

!pip install -q gradio>=4.0 beautifulsoup4 requests mediapipe pillow



In [ ]:
# ── CELL 2: Mount Drive & set paths ──────────────────────────────
"""
from google.colab import drive
drive.mount('/content/drive')
 
import sys
sys.path.insert(0, '/content/SMART-WARDROBE/src')
sys.path.insert(0, '/content/SMART-WARDROBE/demo')
"""


In [ ]:
# ── CELL 3: Quick test — scraper only ────────────────────────────
"""
from scraper import scrape_products, download_product_images
 
products = scrape_products(
    url="https://www.myntra.com/topwear",
    category="topwear",
    max_products=10
)
 
print(f"Scraped {len(products)} products:")
for p in products[:3]:
    print(f"  {p['name'][:50]} | {p['price']} | {p['image_url'][:60]}")
"""


In [ ]:
# ── CELL 4: Quick test — full pipeline (measurements mode) ────────
"""
from recommender import SmartWardrobeRecommender
 
rec = SmartWardrobeRecommender(
    model_path="/content/drive/MyDrive/SmartWardrobe/best_vibe_model.pth"
)
 
results = rec.recommend_from_measurements(
    measurements={
        "height_cm":   165,
        "bust_cm":     88,
        "waist_cm":    70,
        "hip_cm":      96,
        "shoulder_cm": 38,
    },
    website_url="https://www.myntra.com/topwear",
    category="topwear",
    top_k=5,
)
 
print(f"\\nTop {len(results)} recommendations:")
for r in results:
    print(f"  #{r['rank']}  sim={r['similarity']:.3f}  {r['name'][:50]}")
    print(f"       {r['product_url'][:80]}")
"""
 


In [ ]:
# ── CELL 5: Full pipeline with user photo ─────────────────────────
"""
from recommender import SmartWardrobeRecommender
from google.colab import files
 
# Upload your photo
uploaded = files.upload()
photo_path = list(uploaded.keys())[0]
 
rec = SmartWardrobeRecommender(
    model_path="/content/drive/MyDrive/SmartWardrobe/best_vibe_model.pth"
)
 
results = rec.recommend(
    user_image_path=photo_path,
    website_url="https://www.myntra.com/topwear",
    category="topwear",
    top_k=10,
)
 
# Display results
from IPython.display import display, HTML
 
html = "<div style='display:flex; flex-wrap:wrap; gap:12px;'>"
for r in results:
    html += f\"\"\"
    <div style='width:160px; border:1px solid #eee; border-radius:8px; overflow:hidden;'>
      <img src='{r["image_url"]}' style='width:160px; height:200px; object-fit:cover;'>
      <div style='padding:8px;'>
        <div style='font-size:12px; font-weight:600;'>#{r["rank"]} sim={r["similarity"]:.3f}</div>
        <div style='font-size:11px; color:#666; margin-top:4px;'>{r["name"][:40]}</div>
        <div style='font-size:11px; color:#333;'>{r["price"]}</div>
        <a href='{r["product_url"]}' style='font-size:10px;' target='_blank'>View product</a>
      </div>
    </div>\"\"\"
html += "</div>"
display(HTML(html))
"""
 


In [ ]:
# ── CELL 6: Launch Gradio UI (full demo) ─────────────────────────
"""
from gradio_app import launch
launch(share=True)   # prints a public URL — share this for your demo
"""


In [ ]:
# ── CELL 7: Visualise similarity scores ──────────────────────────
"""
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from io import BytesIO
from PIL import Image
 
def show_recommendations(results, cols=5):
    n    = min(len(results), cols)
    fig, axes = plt.subplots(1, n, figsize=(n*3, 4))
    if n == 1: axes = [axes]
 
    for ax, r in zip(axes, results[:n]):
        try:
            resp = requests.get(r["image_url"], timeout=6)
            img  = Image.open(BytesIO(resp.content)).convert("RGB")
            ax.imshow(img)
        except:
            ax.set_facecolor("#eee")
 
        ax.set_title(
            f"#{r['rank']}  {r['similarity']:.3f}\\n{r['name'][:25]}",
            fontsize=9, pad=4
        )
        ax.axis("off")
 
    plt.suptitle("SmartWardrobe — Top Recommendations", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig("recommendations.png", dpi=150, bbox_inches="tight")
    plt.show()
 
show_recommendations(results)
"""
 

